# PROJECT: Fine-tune ResNet-50 on Custom Data

Reach for this when you need: 
- Complete Transfer Learning boilerplate for Computer Vision.
- Reference for discriminative learning rates and unfreezing strategies.
- Implementing standard image classification with a pre-trained backbone.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Data Pipeline
Using ImageNet normalization as the model was pretrained on ImageNet.

In [ ]:
tform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Using CIFAR-10 as a proxy for 'custom data'
train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=tform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

test_ds = datasets.CIFAR10(root='./data', train=False, download=True, transform=tform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

## 2. Model Setup (Transfer Learning)

| Strategy | Action | Purpose |
| :--- | :--- | :--- |
| **Freezing** | `requires_grad = False` | Faster initial training, preserves learned features |
| **Head Replace** | `model.fc = Linear(...)` | Adapting to custom class count |
| **Unfreezing** | `requires_grad = True` | Final optimization of feature extractors |

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Phase 1: Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace head
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

## 3. Fine-tuning with Discriminative Learning Rates

✅ **Use when**: After the head is converged, update the backbone slowly to adapt to specific textures of new data.
❌ **Don't use when**: Data is too small (overfitting) or very similar to ImageNet.

In [ ]:
# Phase 2: Unfreeze bottom layers
for param in model.layer4.parameters():
    param.requires_grad = True

# Discriminative Learning Rates: Layer4 (lower LR), Head (higher LR)
optimizer = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-2)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

### Common Pitfalls
- **Normalization**: Forgetting ImageNet normalization (`mean=[0.485, ...]`) reduces accuracy even if the head is trained well.
- **LR Scale**: Setting a high LR on the backbone during unfreezing can destroy pretrained features ("Catastrophic Forgetting").
- **Batching**: Large images (224x224) consume significant VRAM; use `accumulation_steps` if OOM.

### Key Takeaways
- Standard fine-tuning involves training a new head first, then unfreezing layers gradually.
- Discriminative Learning Rates ensure the head adapts quickly while the backbone updates gently.
- Always log validation accuracy to detect the point where fine-tuning starts to overfit.